# FraudShield — Model Training\nRun this notebook from the extracted project root. It trains a model separately from the frontend and saves it to `models/fraud_model.joblib`.\n

In [ ]:
from pathlib import Path\nimport os, sys\nROOT = Path.cwd()\nif not (ROOT / 'backend' / 'config.py').exists():\n    raise RuntimeError('Open Jupyter with the FraudShield_Project folder as the working directory.')\nif str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))\nprint('Project root:', ROOT)\n

In [ ]:
import pandas as pd\ndf = pd.read_csv(ROOT / 'data' / 'credit_card_fraud_detection_2026.csv')\nprint('Shape:', df.shape)\ndisplay(df.head())\ndisplay(df['is_fraud'].value_counts().rename('rows').to_frame())\ndisplay((df['is_fraud'].value_counts(normalize=True) * 100).round(2).rename('percent').to_frame())\n

In [ ]:
from sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.ensemble import RandomForestClassifier\nimport joblib\nfrom backend.config import FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES\ndata = df[FEATURES + ['is_fraud']].dropna(subset=['is_fraud']).copy()\nX, y = data[FEATURES], data['is_fraud'].astype(int)\nnumeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])\ncategorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])\npreprocess = ColumnTransformer([('numeric', numeric_pipe, NUMERIC_FEATURES), ('categorical', categorical_pipe, CATEGORICAL_FEATURES)])\nmodel = Pipeline([('preprocessor', preprocess), ('classifier', RandomForestClassifier(n_estimators=250, min_samples_leaf=2, class_weight='balanced_subsample', random_state=42, n_jobs=-1))])\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\nmodel.fit(X_train, y_train)\npred = model.predict(X_test)\nproba = model.predict_proba(X_test)[:, 1]\nprint(classification_report(y_test, pred, digits=4, zero_division=0))\nprint('Confusion matrix [[TN, FP], [FN, TP]]:\\n', confusion_matrix(y_test, pred))\nprint('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))\nprint('PR-AUC:', round(average_precision_score(y_test, proba), 4))\njoblib.dump({'pipeline': model, 'features': FEATURES, 'threshold': 0.5, 'model_name': 'Random Forest'}, ROOT / 'models' / 'fraud_model.joblib')\nprint('Saved model to', ROOT / 'models' / 'fraud_model.joblib')\n